# High-resolution saliency figure (about 10–15 min)
Regenerates the saliency figure of the paper (Grad-CAM, Grad-CAM++ and, for the hybrid model, attention) **directly from the
trained models of the initial run**, at full image resolution (768 × 768 px tiles) and 600 dpi, with a perceptually uniform
colour scale (viridis). The models, test split and saliency code are identical to notebook 3; only the figure is new.

**How to run (Google Colab):** *Runtime → Change runtime type → T4 GPU* (CPU also works, slower). Then *Runtime → Run all*.
When asked, choose **unified_results.zip** (the same file used for notebook 3). At the end `saliency_figure.zip` is
downloaded (main figure and a supplementary version with three tiles per class, as PDF, PNG and TIFF). **Send the zip to Claude.**

In [ ]:
import os
TEST_MODE = os.environ.get("XAI_TEST_MODE") == "1"
SEED, IMG_SIZE = 42, 160
N_SUPP = 3                # tiles per class in the supplementary version
RESULTS_DIR, OUT_DIR = "/content/unified_results", "/content/saliency_figure"
DATA_DIR = None           # None = download LC25000 via kagglehub
N_TEST = None             # None = full shared test split
if TEST_MODE:
    RESULTS_DIR, OUT_DIR, DATA_DIR = os.environ["XAI_RESULTS"], os.environ["XAI_OUT"], os.environ["XAI_DATA"]
    N_TEST = int(os.environ.get("XAI_NTEST", "60"))

In [ ]:
import json, glob, shutil, zipfile, hashlib
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf, keras
from keras import layers
os.makedirs(OUT_DIR, exist_ok=True)
CLASSES = ["lung_aca", "lung_n", "lung_scc"]
NAMES = ["MobileNetV2", "EfficientNetB0", "Hybrid_EffNetB0_Transformer"]
SHORT = {"MobileNetV2": "MobileNetV2", "EfficientNetB0": "EfficientNetB0", "Hybrid_EffNetB0_Transformer": "Hybrid"}

In [ ]:
def have(d): return all(os.path.exists(os.path.join(d, n, "best.weights.h5")) for n in NAMES) and os.path.exists(os.path.join(d, "split_test.csv"))
if not have(RESULTS_DIR):
    from google.colab import files
    print("Please choose unified_results.zip:"); up = files.upload()
    os.makedirs(RESULTS_DIR, exist_ok=True); zipfile.ZipFile([k for k in up if k.endswith(".zip")][0]).extractall(RESULTS_DIR)
    if not have(RESULTS_DIR):
        RESULTS_DIR = os.path.dirname(glob.glob(RESULTS_DIR + "/**/split_test.csv", recursive=True)[0])
assert have(RESULTS_DIR)

def find_lung_dir(root):
    for d, ds, _ in os.walk(root):
        if all(c in ds for c in CLASSES): return d
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = find_lung_dir(kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images"))
test = pd.read_csv(os.path.join(RESULTS_DIR, "split_test.csv"))
if N_TEST: test = test.groupby("label", group_keys=False).head(N_TEST // 3).reset_index(drop=True)
test["local"] = [os.path.join(DATA_DIR, c, os.path.basename(p)) for c, p in zip(test["cls"], test["path"])]
ok = [hashlib.md5(open(f, "rb").read()).hexdigest() == m for f, m in zip(test["local"], test["md5"])]
assert all(ok) or TEST_MODE, "test images do not match the original split - send this message to Claude"
print("Shared test split restored:", len(test), "images (MD5 verified)")

def load(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.cast(tf.round(tf.image.resize(img, (IMG_SIZE, IMG_SIZE))), tf.uint8).numpy().astype("float32")
X = np.stack([load(f) for f in test["local"]]); y = test["label"].values

In [ ]:
def head(x):
    x = layers.Dropout(0.3, name="dropout")(x)
    return layers.Activation("softmax", name="pred")(layers.Dense(3, name="logits")(x))
def build(name):
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image"); attn = None
    if name == "MobileNetV2":
        x = layers.Rescaling(1 / 127.5, offset=-1, name="mnv2_preprocess")(inp)
        base = keras.applications.MobileNetV2(include_top=False, weights=None, input_tensor=x)
        out = head(layers.GlobalAveragePooling2D(name="gap")(base.output)); cam = "out_relu"
    else:
        base = keras.applications.EfficientNetB0(include_top=False, weights=None, input_tensor=inp); cam = "top_activation"
        if name == "EfficientNetB0":
            out = head(layers.GlobalAveragePooling2D(name="gap")(base.output))
        else:
            f = base.output; h, w, c = f.shape[1], f.shape[2], f.shape[3]
            tok = layers.Reshape((h * w, c), name="tokens")(f); t = layers.LayerNormalization(epsilon=1e-6, name="ln1")(tok)
            a, attn = layers.MultiHeadAttention(num_heads=2, key_dim=16, name="mhsa")(t, t, return_attention_scores=True)
            x = layers.Add(name="res1")([tok, a]); t = layers.LayerNormalization(epsilon=1e-6, name="ln2")(x)
            t = layers.Dense(c, name="mlp2")(layers.Dense(128, activation="gelu", name="mlp1")(t))
            out = head(layers.GlobalAveragePooling1D(name="gap")(layers.Add(name="res2")([x, t])))
    m = keras.Model(inp, out, name=name); m.load_weights(os.path.join(RESULTS_DIR, name, "best.weights.h5"))
    grad_model = keras.Model(m.inputs, [m.get_layer(cam).output, m.get_layer("logits").output])
    attn_model = keras.Model(m.inputs, attn) if attn is not None else None
    return m, grad_model, attn_model

def build_random(name):
    """Same architecture with untrained (random) weights, for the model-randomisation sanity check."""
    global RESULTS_DIR
    saved = keras.Model.load_weights
    keras.Model.load_weights = lambda self, *a, **k: None      # skip loading trained weights
    try: return build(name)
    finally: keras.Model.load_weights = saved

In [ ]:
def gradcam_and_pp(grad_model, img, c):
    with tf.GradientTape() as tape:
        A, logits = grad_model(img[None], training=False); S = logits[:, c]
    g = tape.gradient(S, A)[0].numpy().astype("float64"); A = A[0].numpy().astype("float64")
    # Grad-CAM: channel weights = spatial mean of gradients, followed by ReLU
    gc = np.maximum((A * g.mean((0, 1))).sum(-1), 0)
    # Grad-CAM++ (closed form, Chattopadhay et al. 2018, Eq. 19): alpha = g^2 / (2 g^2 + sum(A) g^3)
    g2, g3 = g ** 2, g ** 3
    denom = 2 * g2 + A.sum((0, 1), keepdims=True) * g3
    alpha = np.where(denom != 0, g2 / np.where(denom != 0, denom, 1), 0)
    w = (alpha * np.maximum(g, 0)).sum((0, 1))
    gpp = np.maximum((A * w).sum(-1), 0)
    return gc, gpp
def attention_map(attn_model, img):
    s = attn_model(img[None], training=False).numpy()[0]       # heads x queries x keys
    r = s.mean(0).mean(0); n = int(np.sqrt(r.size)); return r.reshape(n, n)

In [ ]:
# %% Tiles correctly classified by all three models (first ones in the shared test split, per class)
MODELS = {n: build(n) for n in NAMES}
pred = {n: MODELS[n][0].predict(X, batch_size=64, verbose=0).argmax(1) for n in NAMES}
allok = np.all([pred[n] == y for n in NAMES], axis=0)
if TEST_MODE: allok = np.ones_like(allok)          # synthetic test images are not classified correctly
SEL = {c: np.where(allok & (y == c))[0][:N_SUPP] for c in range(3)}
for c in range(3):
    assert len(SEL[c]) == N_SUPP, "not enough correctly classified tiles - send this message to Claude"
    print(CLASSES[c], [os.path.basename(test["local"][k]) for k in SEL[c]])

In [ ]:
# %% Saliency maps (each normalised to its own range, then bilinearly upsampled to the full 768 x 768 tile)
def norm(m):
    m = np.asarray(m, float); m = m - m.min(); return m / m.max() if m.max() > 0 else m
def full_image(path):
    return tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False).numpy()
COLS = [("MobileNetV2", "Grad-CAM"), ("MobileNetV2", "Grad-CAM++"), ("EfficientNetB0", "Grad-CAM"), ("EfficientNetB0", "Grad-CAM++"),
        ("Hybrid_EffNetB0_Transformer", "Grad-CAM"), ("Hybrid_EffNetB0_Transformer", "Grad-CAM++"), ("Hybrid_EffNetB0_Transformer", "Attention")]
MAPS = {}
for c in range(3):
    for k in SEL[c]:
        img, row = X[k], {}
        for n in NAMES:
            _, gm, am = MODELS[n]; gc, gpp = gradcam_and_pp(gm, img, c)
            row[n, "Grad-CAM"], row[n, "Grad-CAM++"] = norm(gc), norm(gpp)
            if am is not None: row[n, "Attention"] = norm(attention_map(am, img))
        MAPS[k] = row
np.savez_compressed(os.path.join(OUT_DIR, "saliency_maps.npz"),
                    **{f"{os.path.basename(test['local'][k])}|{n}|{m}": v for k, r in MAPS.items() for (n, m), v in r.items()})
print("maps computed for", len(MAPS), "tiles")

In [ ]:
# %% Figure (600 dpi; viridis colour map, no red)
plt.rcParams.update({"font.family": "DejaVu Serif", "pdf.fonttype": 42})
CMAP, ALPHA = matplotlib.colormaps["viridis"], 0.5
ROWNAME = ["Adenocarcinoma", "Benign", "Squamous"]
def overlay(rgb, m):
    H = rgb.shape[0]; up = tf.image.resize(m[..., None], (H, H), method="bilinear").numpy()[..., 0]
    return np.clip((1 - ALPHA) * rgb / 255.0 + ALPHA * CMAP(np.clip(up, 0, 1))[..., :3], 0, 1)
def make(per_class, fname):
    rows = [(c, k) for c in range(3) for k in SEL[c][:per_class]]
    W = 7.4; cell = (W - 0.75) / 8; Hfig = 0.5 + cell * len(rows) + 0.06 * (len(rows) - 1) + 0.5
    fig = plt.figure(figsize=(W, Hfig))
    for r, (c, k) in enumerate(rows):
        rgb = full_image(test["local"][k]); top = Hfig - 0.5 - r * (cell + 0.06)
        for j in range(8):
            ax = fig.add_axes([(0.45 + j * cell + (0.06 if j > 0 else 0) + (0.06 if j > 2 else 0) + (0.06 if j > 4 else 0) - 0.18 * 0) / W,
                               (top - cell) / Hfig, (cell - 0.04) / W, (cell - 0.04) / Hfig])
            ax.imshow(rgb if j == 0 else overlay(rgb, MAPS[k][COLS[j - 1]]), interpolation="lanczos"); ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values(): s.set_linewidth(0.4); s.set_color("#4b5563")
            if r == 0: ax.set_title("Input" if j == 0 else COLS[j - 1][1], fontsize=7.2, pad=3, color="#1f2937")
            if j == 0 and (per_class == 1 or k == SEL[c][0]):
                ax.set_ylabel(ROWNAME[c], fontsize=7.6, color="#1f2937", labelpad=4)
    # model headers
    for j0, j1, lab in [(1, 2, "MobileNetV2"), (3, 4, "EfficientNetB0"), (5, 7, "Hybrid")]:
        x0 = (0.45 + j0 * cell + 0.06 * (1 + (j0 > 2) + (j0 > 4))) / W; x1 = (0.45 + (j1 + 1) * cell - 0.04 + 0.06 * (1 + (j1 > 2) + (j1 > 4))) / W
        y = (Hfig - 0.16) / Hfig
        fig.add_artist(matplotlib.lines.Line2D([x0, x1], [y - 0.06 / Hfig, y - 0.06 / Hfig], color="#1f2937", lw=0.7))
        fig.text((x0 + x1) / 2, y, lab, ha="center", va="bottom", fontsize=8.6, weight="bold", color="#1f2937")
    cax = fig.add_axes([0.3, 0.18 / Hfig, 0.4, 0.09 / Hfig])
    cb = fig.colorbar(matplotlib.cm.ScalarMappable(cmap=CMAP), cax=cax, orientation="horizontal", ticks=[0, 1])
    cb.ax.set_xticklabels(["Low", "High"], fontsize=7); cb.outline.set_linewidth(0.4)
    cax.set_title("Normalised saliency", fontsize=7.2, pad=2)
    for ext in ["pdf", "png"]: fig.savefig(os.path.join(OUT_DIR, f"{fname}.{ext}"), dpi=600)
    plt.close(fig)
    from PIL import Image
    Image.open(os.path.join(OUT_DIR, f"{fname}.png")).convert("RGB").save(os.path.join(OUT_DIR, f"{fname}.tif"), compression="tiff_lzw", dpi=(600, 600))
make(1, "fig_xai_compare"); make(N_SUPP, "fig_xai_compare_supplementary")
json.dump({CLASSES[c]: [os.path.basename(test["local"][k]) for k in SEL[c]] for c in range(3)}, open(os.path.join(OUT_DIR, "tiles_used.json"), "w"), indent=1)
print("figures saved:", sorted(os.listdir(OUT_DIR)))

In [ ]:
# %% Download
shutil.make_archive(OUT_DIR.rstrip("/"), "zip", OUT_DIR)
print("=" * 60 + "\nDONE - send saliency_figure.zip to Claude\n" + "=" * 60)
if not TEST_MODE:
    from google.colab import files; files.download(OUT_DIR.rstrip("/") + ".zip")